# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第10章：音声知覚データ（1）：同定課題・カテゴリ境界・手がかりの重みづけ


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Idemaru & Holt (2011), *Journal of Experimental Psychology: Human Perception and Performance*／Liberman, Harris, Hoffman & Griffith (1957) 以来のカテゴリ知覚研究。

In [ ]:
install.packages(c("lme4", "lmerTest"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ 知覚データは産出データと何が違うか：

第1〜9章で扱ってきたVOTやF0は**産出（production）**のデータ、つまり話者が出した音を機械が測った値である。これに対して**知覚（perception）**のデータは、聞き手がどう聞こえたかを答えた結果であり、性質が3点違う。

第一に、応答が連続量ではない。「/pa/ に聞こえたか /ba/ に聞こえたか」という二択（あるいは7段階評定などの順序）であり、そのまま平均を取る対象ではない。第二に、刺激が実験者の作った連続体であることが多く、**刺激そのものが独立変数として設計されている**。第三に、同じ聞き手が数百試行を繰り返すため、聞き手のランダム効果が産出データ以上に効く。

**実際の研究例**：Idemaru & Holt (2011) は、VOTとF0という2つの音響手がかりの間の相関を実験的に操作し（人工的な「アクセント」を模した曝露）、聞き手がその相関に応じて手がかりの重みづけを短時間で調整する（統計的学習）ことを示した。本章で作る例は、この研究そのものの再現ではなく、**そこで前提となっているVOTとF0の複合的な手がかり利用**（英語話者にとってVOTが第一の手がかり、F0は補助的だが確かに効くという基本構図）に着想を得た単純化例である。実際の相関操作・学習過程は再現していない。本章では、この単純化した同定課題のデータを作り、**カテゴリ境界**と**手がかりの重み**という知覚研究の2つの中心的な指標を推定する。

■ 同定課題データの生成：

In [ ]:
set.seed(2024)

n_listener <- 24                        # 聞き手24名
vot_steps  <- seq(10, 70, by = 10)      # VOT連続体を7段階（10〜70ms）で合成した刺激
f0_levels  <- c("low", "high")          # 後続母音の開始F0を2水準
n_rep      <- 8                         # 各刺激を8回ずつ提示

design <- expand_grid(
  listener = sprintf("L%02d", 1:n_listener),
  vot = vot_steps,
  f0  = f0_levels,
  rep = 1:n_rep
)

# ---- 真のパラメータ（あとで推定値と見比べる）----
b0    <- -3.2    # 切片
b_vot <-  0.09   # VOTが1ms増えるごとにログオッズが0.09上がる
b_f0  <-  0.8    # F0が高いと「無声」と答えやすくなる（Idemaru & Holtの補助手がかり）

listener_intercept <- rnorm(n_listener, 0, 0.7)   # 聞き手ごとの反応の偏り
listener_slope     <- rnorm(n_listener, 0, 0.03)  # 聞き手ごとのVOT感受性の差

perc <- design %>%
  mutate(
    li = as.integer(factor(listener)),
    # F0は±0.5の対比にしておく（第4章のsum coding。切片が2水準の中間になる）
    f0_num = if_else(f0 == "high", 0.5, -0.5),
    eta = b0 + (b_vot + listener_slope[li]) * vot + b_f0 * f0_num + listener_intercept[li],
    # response = 1 は「無声(/pa/)と答えた」を表す
    response = rbinom(n(), size = 1, prob = plogis(eta))
  ) %>%
  select(-li, -eta)

cat("総試行数:", nrow(perc), "\n")

# 同定率の表。VOTが長いほど「無声」の回答が増えるはず。
perc %>%
  group_by(vot, f0) %>%
  summarise(p_voiceless = round(mean(response), 3), .groups = "drop") %>%
  pivot_wider(names_from = f0, values_from = p_voiceless)

# --- 実行結果 ---
# 総試行数: 2688
#
# # A tibble: 7 x 3
#     vot  high   low
#   <dbl> <dbl> <dbl>
# 1    10 0.177 0.047
# 2    20 0.234 0.151
# 3    30 0.448 0.333
# 4    40 0.661 0.49
# 5    50 0.734 0.63
# 6    60 0.792 0.755
# 7    70 0.896 0.792
#
# 【出力の読み方】
# ・表の各セルが「その刺激で『無声』と答えた割合」である。
#   どの列も上から下へ単調に増えており、**VOTが長いほど無声と聞こえる**という
#   期待どおりの形になっている。この単調性が崩れていたら、刺激の合成か
#   データの並びを疑う。
# ・列を左右で比べると、**同じVOTでもhighのほうが値が大きい**。
#   たとえばVOT=40msでは high 0.661 に対し low 0.490。
#   F0が高いだけで「無声」と答える割合が17ポイント上がっている。
#   これがIdemaru & Holt (2011) の言う補助的手がかりの効果である。
# ・0.5を挟む位置に注目する。lowでは40ms付近（0.490）で0.5をまたぐが、
#   highでは30〜40msの間（0.448→0.661）で既にまたいでいる。
#   **F0が高いほうがカテゴリ境界が短いVOT側へ動く**ことが表から読み取れる。
#   この「境界の移動量」を数値として推定するのが本章の目的である。
# ・端の値が0や1に達していないこと（10msで0.047、70msで0.896）も確認する。
#   両端が完全に0と1に張り付いていると、境界と傾きの推定が不安定になる。
#   **連続体の範囲が狭すぎないか**を確かめる意味がある。


■ 同定曲線を描く：

In [ ]:
ggplot(perc, aes(x = vot, y = response, color = f0)) +
  # stat_summary で各刺激点の実測比率を点として打つ
  stat_summary(fun = mean, geom = "point", size = 2.5) +
  # method.args で二項分布を指定し、ロジスティック曲線を当てはめて重ねる
  geom_smooth(method = "glm", method.args = list(family = binomial),
              se = TRUE, linewidth = 0.9) +
  geom_hline(yintercept = 0.5, linetype = "dashed", color = "grey50") +
  labs(title = "VOT連続体の同定曲線（F0別）",
       x = "VOT (ms)", y = "「無声」と答えた割合") +
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果 ---
# S字カーブが2本描かれた図が1枚出る。テキスト出力は無い。
#
# 【図の読み方】
# ・縦軸は「無声」と答えた割合、横軸は刺激のVOT。点が実測値、曲線が
#   ロジスティック関数の当てはめ、帯が95%信頼区間である。
# ・**S字（シグモイド）になっていることが、カテゴリ知覚の基本的な証拠**である。
#   直線的に増えるのではなく、両端では変化が緩く、境界付近で急に切り替わる。
#   もし直線に近ければ、聞き手はカテゴリではなく連続的な程度として
#   聞いている可能性がある。
# ・破線（y = 0.5）と曲線が交わる横軸の値が**カテゴリ境界**である。
#   2本の曲線は破線と別々の位置で交わり、**F0がhighの曲線のほうが左**にある。
#   つまり境界が短いVOT側へ移動している。
# ・曲線の**傾きの急さ**がもう1つの重要な指標で、急なほど
#   カテゴリの区別が明確であることを意味する。第二言語学習者は
#   母語話者より曲線が緩やかになることが多く、この傾きが習得の指標として使われる。
# ・★この図の geom_smooth() は**聞き手をひとまとめにした当てはめ**である。
#   図として概観を掴むにはよいが、**このまま統計的な結論を出してはいけない**。
#   聞き手ごとのばらつきを無視しているためで、次のセルでGLMMを使う。


■ GLMMでカテゴリ境界と手がかりの重みを推定する：

同定課題の応答は0/1なので、第6章で扱ったロジスティック混合効果モデルがそのまま使える。違いは、予測子が実験者の設計した連続量（VOT）である点である。

In [ ]:
model_id <- glmer(
  # vot: 連続量としてそのまま入れる（段ごとの因子にしない。理由は下の解説）
  # f0_num: ±0.5の対比
  # (1 + vot | listener): 聞き手ごとに反応の偏りとVOT感受性の両方を許す
  response ~ vot + f0_num + (1 + vot | listener),
  family = binomial,
  data = perc
)

summary(model_id)

fe <- fixef(model_id)

# カテゴリ境界 = 予測確率が0.5になるVOT = -(切片) / (VOTの係数)
cat("\nカテゴリ境界（F0が中間のとき）:", round(-fe["(Intercept)"] / fe["vot"], 2), "ms\n")
cat("境界 F0 = low  :", round(-(fe["(Intercept)"] + fe["f0_num"] * (-0.5)) / fe["vot"], 2), "ms\n")
cat("境界 F0 = high :", round(-(fe["(Intercept)"] + fe["f0_num"] * ( 0.5)) / fe["vot"], 2), "ms\n")

# 手がかりの重みを共通の単位で比べる: F0の効果は「VOT何ms分」に相当するか
cat("F0を低→高に変える効果は VOT", round(fe["f0_num"] / fe["vot"], 2), "ms 分に相当\n")

# --- 実行結果（抜粋）---
#             Estimate Std. Error  z value Pr(>|z|)
# (Intercept)  -3.1280     0.1908 -16.3904        0
# vot           0.0837     0.0066  12.6881        0
# f0_num        0.7464     0.1038   7.1913        0
#
# カテゴリ境界（F0が中間のとき）: 37.36 ms
# 境界 F0 = low  : 41.82 ms
# 境界 F0 = high : 32.9 ms
# F0を低→高に変える効果は VOT 8.91 ms 分に相当
#
# 【出力の読み方】
# ・まず推定値が生成時の値を回収できているか確かめる。
#   切片は −3.128（真値 −3.2）、VOTの係数は 0.0837（真値 0.09）、
#   F0は 0.7464（真値 0.8）。いずれも近い。**擬似データの利点は、
#   このように「正解」と見比べられること**にある。
# ・vot の係数 0.0837 は「VOTが1ms長くなるとログオッズが0.0837上がる」。
#   ログオッズのままでは実感できないので、次の2つに直して報告する。
# ・**カテゴリ境界**は、予測確率が0.5になるVOTである。ログオッズが0になる
#   点なので、切片 + 係数×VOT = 0 を解いて VOT = −切片/係数 となる。
#   ここでは 3.128/0.0837 = 37.36ms。**この値は刺激の範囲（10〜70ms）の
#   内側にある**ことを必ず確認する。外に出ていたら、連続体が
#   境界をまたいでいないということであり、刺激の設計を見直す必要がある。
# ・**F0によって境界が 41.82ms から 32.90ms へ約8.9ms動いている。**
#   これがF0という補助手がかりの効果の大きさである。
# ・最後の行が**手がかりの重みづけ**の考え方である。VOTとF0は単位が違う
#   （msとF0水準）ので係数を直接比べても意味がない。そこで
#   「F0の係数 ÷ VOTの係数」を計算し、**F0の効果をVOT換算のミリ秒**で表す。
#   8.91msという値は、Idemaru & Holtが示したような「VOTが主、F0が従」という
#   関係を1つの数値で言い表したものである。この換算を**トレーディング関係**
#   （trading relation）と呼ぶ。
# ・★境界も重みも**係数の比**であることに注意する。比の標準誤差は
#   自動では出ないので、信頼区間が必要ならデルタ法かブートストラップを使う。
#   「境界は37.36msだった」と点推定だけ書いて済ませるのは不十分である。
# ・★vot を因子にしてはいけない理由: 7段階を factor(vot) にすると、
#   段どうしの順序と等間隔性という情報が捨てられ、係数が6本に増えて
#   境界も傾きも定義できなくなる。**連続体は連続量として扱う。**


■ ★よくある誤り：聞き手ごとに境界を出してから検定する（2段階法）：

知覚研究で長く使われてきたのが、**聞き手1人ずつに曲線を当てはめて境界を求め、その境界の値を条件間でt検定する**という2段階の手続きである。直感的で分かりやすいが、GLMMと比べると2つの問題を抱えている。実際に同じデータで両方を計算して比べる。

In [ ]:
# ---- 誤り1: 「割合が初めて0.5を超えた段」を境界とする ----
naive <- perc %>%
  group_by(listener, vot) %>%
  summarise(p = mean(response), .groups = "drop") %>%
  group_by(listener) %>%
  summarise(boundary = {
    idx <- which(p >= 0.5)
    if (length(idx) == 0) NA_real_ else vot[min(idx)]
  }, .groups = "drop")

cat("素朴な境界: 平均", round(mean(naive$boundary, na.rm = TRUE), 2),
    " SD", round(sd(naive$boundary, na.rm = TRUE), 2), "\n")
cat("  取りうる値:", paste(sort(unique(naive$boundary)), collapse = ", "), "\n")

# ---- 誤り2: 聞き手ごとにglm()を当てはめ、境界を出す（2段階法）----
per_listener <- perc %>%
  group_by(listener) %>%
  group_modify(~{
    fit <- glm(response ~ vot + f0_num, family = binomial, data = .x)
    cf <- coef(fit)
    tibble(boundary = -cf[1] / cf[2], slope = cf[2])
  }) %>%
  ungroup()

cat("\n2段階法の境界: 平均", round(mean(per_listener$boundary), 2),
    " SD", round(sd(per_listener$boundary), 2), "\n")
cat("  範囲:", round(min(per_listener$boundary), 1), "-",
    round(max(per_listener$boundary), 1), "ms\n")
cat("  刺激の範囲は 10 - 70 ms であることに注意\n")

cat("\nGLMMの境界   :", round(-fixef(model_id)["(Intercept)"] / fixef(model_id)["vot"], 2),
    "ms（全2688試行を使った1つの推定値）\n")

# --- 実行結果 ---
# 素朴な境界: 平均 39.55  SD 13.27
#   取りうる値: 20, 30, 40, 50, 60, 70
#
# 2段階法の境界: 平均 42.08  SD 19.55
#   範囲: 21.8 - 102.7 ms
#   刺激の範囲は 10 - 70 ms であることに注意
#
# GLMMの境界   : 37.36 ms（全2688試行を使った1つの推定値）
#
# 【出力の読み方】
# ・**誤り1の致命的な点は、境界が刺激の段の値しか取れない**ことである。
#   「取りうる値: 20, 30, 40, 50, 60, 70」のとおり、7段階の刺激で実験すれば
#   境界も7通りの値にしかならない。実際の境界が37.4msであっても、
#   この方法では30msか40msとしか答えられない。**測定の分解能が
#   刺激の粗さで決まってしまう。** SDの13.27も、聞き手の個人差ではなく
#   段の粗さを主に反映している。
# ・**誤り2の致命的な点は、外挿が起きる**ことである。境界の範囲が
#   21.8〜102.7msとなっているが、**102.7msという刺激は提示していない**
#   （連続体は10〜70ms）。この聞き手は最後まで50%を越えなかったか、
#   曲線が非常に緩やかだったため、モデルが範囲の外まで直線を伸ばして
#   境界を「推定」してしまった。この値をそのまま平均に含めると、
#   全体の平均（42.08ms）が上方に歪む。
# ・2段階法のSD 19.55 は、GLMMのランダム効果が捉える真の個人差よりも大きい。
#   **1人あたり112試行しかないので、各人の推定値自体が誤差を含む**。
#   その誤差が「個人差」として二重に数えられている。
# ・GLMMが優れているのは、**部分プーリング（partial pooling）**を行うためである。
#   データの少ない聞き手の推定値は全体の平均へ引き寄せられ、
#   極端な外挿が抑制される。同時に、試行レベルの情報を捨てないので
#   刺激の粗さにも縛られない。
# ・★ただし2段階法が常に間違いというわけではない。試行数が各聞き手で十分多く、
#   全員の曲線が連続体の内側で0.5を横切っているなら、結果はGLMMに近づく。
#   **使ってよいかどうかは、上のような診断（外挿の有無・段の粗さ）を
#   自分で確認して判断する。** 「先行研究がそうしているから」は理由にならない。


■ 演習：

1. `f0_num` を含めずに `response ~ vot + (1 + vot | listener)` を当てはめ、カテゴリ境界の推定値がどう変わるかを確かめよ。
2. 聞き手ごとのランダム傾き（VOT感受性）を取り出し、感受性が高い聞き手ほど同定曲線が急であることを確認せよ。
3. 刺激の範囲を10〜40msに狭めた部分データで同じモデルを当てはめ、境界の推定がどうなるかを観察せよ。

■ 演習の解答：

3問目は、連続体の設計が推定を左右することを体験する問題である。

In [ ]:
# 1. F0を落としたモデルと比べる。
model_no_f0 <- glmer(response ~ vot + (1 + vot | listener),
                     family = binomial, data = perc)
fe2 <- fixef(model_no_f0)
cat("F0あり の境界:", round(-fixef(model_id)["(Intercept)"] / fixef(model_id)["vot"], 2), "ms\n")
cat("F0なし の境界:", round(-fe2["(Intercept)"] / fe2["vot"], 2), "ms\n")

# 2. 聞き手ごとのVOT感受性（ランダム傾き）を取り出す。
re <- ranef(model_id)$listener
slopes <- fixef(model_id)["vot"] + re[, "vot"]
cat("\n聞き手ごとのVOT係数: 最小", round(min(slopes), 4),
    " 最大", round(max(slopes), 4), "\n")
cat("最も感受性が高い聞き手は、最も低い聞き手の",
    round(max(slopes) / min(slopes), 2), "倍の傾き\n")

# 3. 連続体を10〜40msに狭めるとどうなるか。
perc_narrow <- perc %>% filter(vot <= 40)
model_narrow <- glmer(response ~ vot + f0_num + (1 + vot | listener),
                      family = binomial, data = perc_narrow)
fe3 <- fixef(model_narrow)
cat("\n刺激を10-40msに狭めた場合:\n")
cat("  VOTの係数:", round(fe3["vot"], 4), " (全範囲では",
    round(fixef(model_id)["vot"], 4), ")\n")
cat("  境界      :", round(-fe3["(Intercept)"] / fe3["vot"], 2), "ms\n")

# --- 実行結果 ---
# F0あり の境界: 37.36 ms
# F0なし の境界: 37.34 ms
#
# 聞き手ごとのVOT係数: 最小 0.034  最大 0.1204
# 最も感受性が高い聞き手は、最も低い聞き手の 3.55 倍の傾き
#
# 刺激を10-40msに狭めた場合:
#   VOTの係数: 0.0904  (全範囲では 0.0837 )
#   境界      : 36.43 ms
#
# 【出力の読み方】
# ・1問目: F0を落としても全体の境界は 37.36 → 37.34ms とほとんど動かない。
#   **これは「F0に効果が無い」という意味ではない。** F0はlowとhighを
#   同数提示した釣り合った設計なので、両者の効果が打ち消し合い、
#   全体の平均的な境界は変わらないのである。F0の効果は
#   「境界が条件によってどれだけ動くか」（41.82ms対32.90ms）として現れるので、
#   **平均だけを見ていると補助手がかりの存在を見落とす。**
#   ★釣り合っていない設計（highを多く提示するなど）では、
#   F0を落とすと境界そのものが偏る。落とした変数が結果に効くかどうかは、
#   設計の釣り合いによって変わる。
# ・2問目: 聞き手ごとのVOT係数は 0.034 から 0.1204 まで、**3.55倍の開き**がある。
#   係数が大きい聞き手ほど同定曲線が急、つまり有声/無声の区別が明確である。
#   同じ刺激を聞いても、聞き手によってカテゴリの鋭さがこれだけ違う。
#   **この個人差こそが (1 + vot | listener) が捉えているもの**であり、
#   ランダム傾きを入れずに全員をひとまとめにすると消えてしまう情報である。
#   第二言語習得の研究では、この傾きそのものを習得の指標として使う。
# ・3問目: 刺激を10〜40msに狭めても、境界は 37.36 → 36.43ms とさほど動かない。
#   **境界(36〜37ms)が、狭めた範囲(10〜40ms)の内側に残っているため**である。
#   VOTの係数は 0.0837 → 0.0904 とやや急になったが、これは
#   曲線の中央付近だけを見ているので傾きが強めに出るためである。
# ・★もし連続体を10〜30msに狭めていたら、境界は範囲の外になり、
#   推定は外挿になって非常に不安定になる。**刺激の設計段階で
#   「境界が範囲の真ん中あたりに来るか」を予備実験で確かめる**必要がある。
#   これは統計の問題ではなく実験計画の問題だが、
#   **設計を誤ると、どれだけ良い統計手法を使っても救えない。**
